# 04: Cold-start fold-in (Phase 4) and personas (Phase 5)

A new user is not in the trained model. Fold-in freezes the ranking model and learns one 32-number
user vector from their 5-10 picks; that vector then scores every movie. Uses the ranking model from
notebook 03b (popularity-weighted negatives, ADR 0002).

Runs locally (repo paths) or in Colab (Drive paths).

In [1]:
import json
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/tastelet")
    PROCESSED_DIR = DRIVE_DIR / "data/processed"
    RESULTS_PATH = DRIVE_DIR / "results/results.csv"
    MODEL_PATH = DRIVE_DIR / "artifacts/ncf_model.pt"
else:
    REPO_DIR = Path.cwd().parent  # notebooks/ -> repo root
    PROCESSED_DIR = REPO_DIR / "data/processed"
    RESULTS_PATH = REPO_DIR / "results/results.csv"
    MODEL_PATH = REPO_DIR / "backend/artifacts/ncf_model.pt"

SEED = 42

train_df = pd.read_csv(PROCESSED_DIR / "ratings_train.csv")
test_df = pd.read_csv(PROCESSED_DIR / "ratings_test.csv")
movie2idx = {int(k): v for k, v in json.loads((PROCESSED_DIR / "movie2idx.json").read_text()).items()}
movies_meta = json.loads((PROCESSED_DIR / "movies_meta.json").read_text())["movies"]

# One row per model index, so titles/genres line up with score arrays.
movies_by_idx = (
    pd.DataFrame(movies_meta)
    .assign(movie_idx=lambda df: df["id"].map(movie2idx))
    .set_index("movie_idx")
    .sort_index()
)
print(train_df.shape, test_df.shape, len(movies_by_idx))

(72088, 6) (9012, 6) 3636


In [2]:
class NCF(nn.Module):
    def __init__(self, n_users: int, n_movies: int, dim: int, dropout: float) -> None:
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.movie_embedding = nn.Embedding(n_movies, dim)
        self.mlp = nn.Sequential(
            nn.Linear(2 * dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def score_vec(self, user_vecs: torch.Tensor, movie_ids: torch.Tensor) -> torch.Tensor:
        # Takes user vectors directly, so a new (folded-in) user can be scored too.
        pair = torch.cat([user_vecs, self.movie_embedding(movie_ids)], dim=1)
        return self.mlp(pair).squeeze(-1)

    def forward(self, user_ids: torch.Tensor, movie_ids: torch.Tensor) -> torch.Tensor:
        return self.score_vec(self.user_embedding(user_ids), movie_ids)


checkpoint = torch.load(MODEL_PATH)
model = NCF(checkpoint["n_users"], checkpoint["n_movies"], checkpoint["embedding_dim"], checkpoint["dropout"])
model.load_state_dict(checkpoint["state_dict"])
model.eval()
n_movies = checkpoint["n_movies"]
all_movie_ids = torch.arange(n_movies)
print(checkpoint["n_users"], n_movies)

610 3636


`score_vec` must give exactly what `forward` gives for a trained user.

In [3]:
users = torch.tensor([0, 1, 2])
movies = torch.tensor([10, 20, 30])
with torch.no_grad():
    via_forward = model(users, movies)
    via_vec = model.score_vec(model.user_embedding(users), movies)
print(via_forward, torch.equal(via_forward, via_vec))

tensor([1.9347, 0.6013, 0.7254]) True


## Fold-in

Start from the average trained user vector and optimize only that vector (all weights frozen) so
picks score 5.0 and some unpicked movies score 0.5, the label the ranking model learned for
negatives. Those negatives are sampled like the training negatives (∝ rating count^0.75), so
popular movies outside the user's taste get pushed down. A small L2 term keeps the vector in the
range of real users. The seeded `rng` makes the same picks give the same result (spec F4).

In [4]:
FOLD_IN_STEPS = 100
FOLD_IN_LR = 0.05
FOLD_IN_NEGATIVES = 20
FOLD_IN_L2 = 1e-3
PICK_RATING = 5.0
NEGATIVE_RATING = 0.5  # must match the label the ranking model was trained with
POPULARITY_EXPONENT = 0.75  # same weighting as the training negatives (ADR 0002)

rating_counts = np.bincount(train_df["movie_idx"], minlength=n_movies)
popularity_weights = rating_counts.astype(float) ** POPULARITY_EXPONENT


def fold_in(
    model: NCF,
    picks: list[int],
    rng: np.random.Generator,
    n_negatives: int = FOLD_IN_NEGATIVES,
    negative_weights: np.ndarray | None = popularity_weights,
) -> torch.Tensor:
    model.eval()
    model.requires_grad_(False)  # weights stay frozen; only the user vector learns
    user_vec = nn.Parameter(model.user_embedding.weight.mean(dim=0, keepdim=True).clone())
    optimizer = torch.optim.Adam([user_vec], lr=FOLD_IN_LR)

    unpicked = np.setdiff1d(np.arange(n_movies), picks)
    probs = None if negative_weights is None else negative_weights[unpicked] / negative_weights[unpicked].sum()
    negatives = rng.choice(unpicked, size=n_negatives, replace=False, p=probs)
    movie_ids = torch.tensor(np.concatenate([picks, negatives]), dtype=torch.long)
    targets = torch.tensor([PICK_RATING] * len(picks) + [NEGATIVE_RATING] * n_negatives)

    for _ in range(FOLD_IN_STEPS):
        optimizer.zero_grad()
        predicted = model.score_vec(user_vec.expand(len(movie_ids), -1), movie_ids)
        loss = ((predicted - targets) ** 2).mean() + FOLD_IN_L2 * (user_vec**2).sum()
        loss.backward()
        optimizer.step()
    return user_vec.detach()

## Ranking

Score all movies with the folded-in vector, drop the picks, keep the top 20. Match score =
predicted score / 5 as a percentage, clipped to 0-100. The ranking model is not a calibrated
star rating, so the match score is a relative strength, not a promise.

In [5]:
N_RECOMMENDATIONS = 20


def score_all_movies(user_vec: torch.Tensor) -> np.ndarray:
    with torch.no_grad():
        return model.score_vec(user_vec.expand(n_movies, -1), all_movie_ids).numpy()


def recommend(user_vec: torch.Tensor, picks: list[int]) -> pd.DataFrame:
    scores = score_all_movies(user_vec)
    scores[picks] = -np.inf  # never recommend a movie the user already picked
    top = np.argsort(-scores, kind="stable")[:N_RECOMMENDATIONS]  # stable: ties break the same way every run
    match_percent = np.clip(scores[top] / PICK_RATING * 100, 0, 100).round().astype(int)
    return movies_by_idx.loc[top, ["title", "year", "genres"]].assign(match=match_percent)

Quick check on a real user's 5 favourite train movies: fits, deterministic, picks excluded.

In [6]:
user_0_picks = train_df[train_df["user_idx"] == 0].nlargest(5, "rating")["movie_idx"].tolist()
print("picks:", movies_by_idx.loc[user_0_picks, "title"].tolist())

user_vec = fold_in(model, user_0_picks, np.random.default_rng(SEED))
print("pick scores after fold-in:", score_all_movies(user_vec)[user_0_picks].round(2))
print("deterministic:", torch.equal(user_vec, fold_in(model, user_0_picks, np.random.default_rng(SEED))))

user_0_recs = recommend(user_vec, user_0_picks)
print("picks in output:", bool(set(user_0_picks) & set(user_0_recs.index)))
user_0_recs

picks: ['Good Morning, Vietnam', 'American History X', 'South Park: Bigger, Longer and Uncut', 'Who Framed Roger Rabbit?', 'Pinocchio']


pick scores after fold-in: [1.31 4.23 5.06 3.53 3.33]
deterministic: True
picks in output: False


,title,year,genres,match
movie_idx,,,,
1386,"Sixth Sense, The",1999,"[Drama, Horror, Mystery]",100
1343,Austin Powers: The Spy Who Shagged Me,1999,"[Action, Adventure, Comedy]",100
1308,"Matrix, The",1999,"[Action, Sci-Fi, Thriller]",100
1025,Saving Private Ryan,1998,"[Action, Drama, War]",98
831,Face/Off,1997,"[Action, Crime, Drama, Thriller]",94
1048,Lady and the Tramp,1955,"[Animation, Children, Comedy, Romance]",93
231,Forrest Gump,1994,"[Comedy, Drama, Romance, War]",93
2078,"Monsters, Inc.",2001,"[Adventure, Animation, Children, Comedy, Fantasy]",93
628,Star Wars: Episode V - The Empire Strikes Back,1980,"[Action, Adventure, Sci-Fi]",93


## Sanity check: three hand-made users

Very different tastes should give clearly different, fitting lists (Phase 4 done-when). Compared
with popularity-weighted negatives (shipped), uniform negatives and no negatives.

In [7]:
def find_movie_idx(title: str, year: int) -> int:
    found = movies_by_idx[(movies_by_idx["title"] == title) & (movies_by_idx["year"] == year)]
    if len(found) != 1:
        raise ValueError(f"{title} ({year}): {len(found)} matches")
    return int(found.index[0])


HAND_MADE_USERS = {
    "animation": [("Toy Story", 1995), ("Lion King, The", 1994), ("Aladdin", 1992),
                  ("Finding Nemo", 2003), ("Monsters, Inc.", 2001)],
    "horror": [("Shining, The", 1980), ("Exorcist, The", 1973), ("Halloween", 1978),
               ("Nightmare on Elm Street, A", 1984), ("Texas Chainsaw Massacre, The", 1974)],
    "crime_90s": [("Pulp Fiction", 1994), ("Goodfellas", 1990), ("Reservoir Dogs", 1992),
                  ("Heat", 1995), ("Casino", 1995)],
}
TASTE_GENRE = {"animation": "Animation", "horror": "Horror", "crime_90s": "Crime"}
hand_made_picks = {
    name: [find_movie_idx(title, year) for title, year in movies] for name, movies in HAND_MADE_USERS.items()
}

In [8]:
FOLD_IN_SETUPS = {
    "popular": {"negative_weights": popularity_weights},
    "uniform": {"negative_weights": None},
    "none": {"n_negatives": 0},
}

hand_made_rows, hand_made_lists = [], {}
for setup, kwargs in FOLD_IN_SETUPS.items():
    lists = {}
    for name, picks in hand_made_picks.items():
        lists[name] = recommend(fold_in(model, picks, np.random.default_rng(SEED), **kwargs), picks)
    row = {
        "negatives": setup,
        "pairwise_overlap": np.mean([len(set(lists[a].index) & set(lists[b].index)) for a, b in combinations(lists, 2)]),
    }
    for name, recs in lists.items():
        row[f"{name}_genre_hits"] = int(recs["genres"].apply(lambda genres: TASTE_GENRE[name] in genres).sum())
    hand_made_rows.append(row)
    hand_made_lists[setup] = lists

pd.DataFrame(hand_made_rows).round(2)

,negatives,pairwise_overlap,animation_genre_hits,horror_genre_hits,crime_90s_genre_hits
0,popular,1.33,7,3,9
1,uniform,4.00,1,1,9
2,none,2.00,3,3,9


In [9]:
for name, recs in hand_made_lists["popular"].items():
    print(f"{name}:")
    print(recs.head(10).to_string(), "\n")

animation:
                                                                                             title  year                                                      genres  match
movie_idx                                                                                                                                                                  
1965                                                                                         Shrek  2001  [Adventure, Animation, Children, Comedy, Fantasy, Romance]     99
2548                                                                                       Shrek 2  2004  [Adventure, Animation, Children, Comedy, Musical, Romance]     81
369                                                                           Beauty and the Beast  1991      [Animation, Children, Fantasy, Musical, Romance, IMAX]     80
2082       Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone)  2001                         

## Cold-start Hit@10

For 100 test users (seed 42) with at least one liked test movie: picks = their 5 highest-rated
train movies, fold in, Hit@10 against their liked test movies (train movies excluded, as in
notebook 02). Compared on the same 100 users with popularity and with the user's trained embedding
(full history, an upper bound).

In [10]:
TOP_K = 10
LIKED_RATING = 4.0
N_COLD_START_USERS = 100
N_COLD_START_PICKS = 5


def hit_at_10(score_fn, train_df: pd.DataFrame, test_df: pd.DataFrame) -> float:
    """Share of test users with at least one liked test movie in their top 10."""
    seen_by_user = train_df.groupby("user_idx")["movie_idx"].apply(set)
    liked = test_df[test_df["rating"] >= LIKED_RATING]
    liked_by_user = liked.groupby("user_idx")["movie_idx"].apply(set)

    hits = 0
    for user_idx, liked_movies in liked_by_user.items():
        scores = np.asarray(score_fn(user_idx), dtype=float).copy()
        seen = list(seen_by_user.get(user_idx, set()))
        scores[seen] = -np.inf  # never recommend a movie already rated in train
        top_k = np.argpartition(-scores, TOP_K)[:TOP_K]
        if liked_movies & set(top_k.tolist()):
            hits += 1
    return hits / len(liked_by_user)


users_with_likes = np.sort(test_df.loc[test_df["rating"] >= LIKED_RATING, "user_idx"].unique())
cold_start_users = np.random.default_rng(SEED).choice(users_with_likes, size=N_COLD_START_USERS, replace=False)
cold_start_test_df = test_df[test_df["user_idx"].isin(cold_start_users)]
cold_start_picks = {
    user_idx: train_df[train_df["user_idx"] == user_idx].nlargest(N_COLD_START_PICKS, "rating")["movie_idx"].tolist()
    for user_idx in cold_start_users
}


def fold_in_scores(**fold_in_kwargs):
    def score(user_idx: int) -> np.ndarray:
        picks = cold_start_picks[user_idx]
        return score_all_movies(fold_in(model, picks, np.random.default_rng(SEED), **fold_in_kwargs))

    return score


def trained_scores(user_idx: int) -> np.ndarray:
    return score_all_movies(model.user_embedding.weight[user_idx : user_idx + 1])


cold_start_rows = [
    {"method": f"fold-in, {setup} negatives", "hit_at_10": hit_at_10(fold_in_scores(**kwargs), train_df, cold_start_test_df)}
    for setup, kwargs in FOLD_IN_SETUPS.items()
]
cold_start_rows += [
    {"method": "popularity", "hit_at_10": hit_at_10(lambda _: rating_counts, train_df, cold_start_test_df)},
    {"method": "trained embedding (full history)", "hit_at_10": hit_at_10(trained_scores, train_df, cold_start_test_df)},
]
cold_start_df = pd.DataFrame(cold_start_rows).round(4)
cold_start_df

,method,hit_at_10
0,"fold-in, popular negatives",0.41
1,"fold-in, uniform negatives",0.39
2,"fold-in, none negatives",0.31
3,popularity,0.36
4,trained embedding (full history),0.50


In [11]:
cold_start_hit_at_10 = cold_start_df.loc[cold_start_df["method"] == "fold-in, popular negatives", "hit_at_10"].item()

results = pd.read_csv(RESULTS_PATH)
results = results[results["model"] != "ncf_cold_start_5picks"].reset_index(drop=True)
results.loc[len(results)] = {"model": "ncf_cold_start_5picks", "rmse": np.nan, "hit_at_10": cold_start_hit_at_10}
results.to_csv(RESULTS_PATH, index=False)
results

,model,rmse,hit_at_10
0,global_mean,1.0334,NaN
1,popularity,NaN,0.3856
2,item_knn,0.8582,0.5441
3,ncf_rating,0.8662,0.1063
4,ncf_ranking_4neg,NaN,0.4072
5,ncf_ranking_popneg,NaN,0.4667
6,ncf_cold_start_5picks,NaN,0.4100
